# Entrenamiento de modelo K-meas con Scikit-learn

In [ ]:
# -*- coding: utf-8 -*-
import os
import re
import random
import warnings
from glob import glob

import joblib
import numpy as np
import pandas as pd
import rasterio
from rasterio.errors import NotGeoreferencedWarning
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.preprocessing import RobustScaler

warnings.filterwarnings("ignore", category=NotGeoreferencedWarning)

In [ ]:
# ==========================
# CONFIGURACIÓN
# ==========================
DATA_DIR = r"C:\SPA_SEI\PROYECTOS\IIAP\Sedimentos\sentinel_1\data_cut_dB"
PATTERN  = re.compile(r"^(\d{8})_multi\.tif$", re.IGNORECASE)

# Muestreo: hasta 5,000 px por imagen, pero adaptativo a cobertura válida
SAMPLE_PER_IMAGE_MAX = 50000
GLOBAL_SAMPLE_MAX = 3_000_000

# Evaluación de K
K_MIN, K_MAX = 5, 8
RANDOM_STATE = 42

# Tamaño máx para calcular métricas (Silhouette/DBI/CH) para no explotar RAM/CPU
METRICS_SAMPLE_MAX = 500_000  # se tomará submuestra si la muestra es mayor

In [ ]:
# ==========================
# UTILIDADES
# ==========================
def list_multis(folder):
    files = sorted(glob(os.path.join(folder, "*_multi.tif")))
    items = []
    for f in files:
        name = os.path.basename(f)
        m = PATTERN.match(name)
        if m:
            date = m.group(1)
            items.append((date, f))
    return items

def read_valid_pixels(path):
    """Lee VV, VH, PR; aplica máscara NoData de la banda 1 (VV) si existe; retorna (N,3) float32."""
    with rasterio.open(path) as src:
        if src.count < 3:
            raise ValueError(f"{os.path.basename(path)} no tiene 3 bandas (VV,VH,PR).")
        vv = src.read(1).astype(np.float32)
        vh = src.read(2).astype(np.float32)
        pr = src.read(3).astype(np.float32)

        nodata = src.nodata  # usamos nodata de VV, por diseño
        if nodata is not None:
            mask = (vv != nodata)
        else:
            # Si no hay nodata, usamos máscara de finitos por seguridad
            mask = np.isfinite(vv) & np.isfinite(vh) & np.isfinite(pr)

        # Asegurar finitos
        mask &= np.isfinite(vv) & np.isfinite(vh) & np.isfinite(pr)

        X = np.stack([vv[mask], vh[mask], pr[mask]], axis=1)
        return X

def sample_rows(X, max_per_image=SAMPLE_PER_IMAGE_MAX):
    """Devuelve submuestra aleatoria de filas de X con tamaño <= max_per_image, sin reemplazo."""
    n = X.shape[0]
    if n == 0:
        return X
    # tamaño adaptativo: no tomar más del 10% si la imagen es muy grande
    target = min(max_per_image, max(1000, int(0.10 * n)))  # al menos 1000 si es posible
    target = min(target, n)
    idx = np.random.choice(n, size=target, replace=False)
    return X[idx]

def choose_best_k(X_scaled, k_min=K_MIN, k_max=K_MAX, random_state=RANDOM_STATE):
    """Evalúa K usando Silhouette (max) y desempata con Davies-Bouldin (min)."""
    # Submuestra para métricas si hace falta
    if X_scaled.shape[0] > METRICS_SAMPLE_MAX:
        idx = np.random.choice(X_scaled.shape[0], METRICS_SAMPLE_MAX, replace=False)
        X_eval = X_scaled[idx]
    else:
        X_eval = X_scaled

    rows = []
    best_row = None

    for k in range(k_min, k_max + 1):
        km = KMeans(n_clusters=k, n_init=10, random_state=random_state)
        labels = km.fit_predict(X_eval)
        sil = silhouette_score(X_eval, labels)
        dbi = davies_bouldin_score(X_eval, labels)
        ch  = calinski_harabasz_score(X_eval, labels)
        rows.append((k, sil, dbi, ch))
        # criterio: silhouette max; empate -> DBI min
        if best_row is None:
            best_row = (k, sil, dbi, ch)
        else:
            _, b_sil, b_dbi, _ = best_row
            if (sil > b_sil) or (np.isclose(sil, b_sil) and (dbi < b_dbi)):
                best_row = (k, sil, dbi, ch)

    metrics_df = pd.DataFrame(rows, columns=["K", "Silhouette", "DaviesBouldin", "CalinskiHarabasz"])
    best_k = best_row[0]
    return best_k, metrics_df

In [ ]:
# ==========================
# MAIN
# ==========================
def main():
    random.seed(RANDOM_STATE)
    np.random.seed(RANDOM_STATE)

    items = list_multis(DATA_DIR)
    if not items:
        raise SystemExit("No se encontraron *_multi.tif en la carpeta de entrada.")

    # Acumular muestra de todas las imágenes
    samples = []
    total_picked = 0

    for date, path in items:
        X = read_valid_pixels(path)
        if X.size == 0:
            print(f"[AVISO] Sin píxeles válidos en {os.path.basename(path)}; se omite de la muestra.")
            continue
        Xs = sample_rows(X, SAMPLE_PER_IMAGE_MAX)
        samples.append(Xs)
        total_picked += Xs.shape[0]
        print(f"[OK] Muestra {date}: {Xs.shape[0]} filas")

    if not samples:
        raise SystemExit("No se logró construir muestra representativa (sin datos válidos).")

    X_all = np.vstack(samples)
    print(f"\nTotal muestreado (antes de tope global): {X_all.shape[0]}")

    # Tope global
    if X_all.shape[0] > GLOBAL_SAMPLE_MAX:
        idx = np.random.choice(X_all.shape[0], GLOBAL_SAMPLE_MAX, replace=False)
        X_all = X_all[idx]
        print(f"[OK] Submuestreado a tope global: {X_all.shape[0]}")

    # Escalado robusto
    scaler = RobustScaler()
    X_scaled = scaler.fit_transform(X_all)

    # Selección de K
    best_k, metrics_df = choose_best_k(X_scaled, K_MIN, K_MAX, RANDOM_STATE)
    print(f"\n[K elegido] {best_k}")

    # Entrenar KMeans final con TODO el X_scaled (no solo submuestra de métricas)
    kmeans = KMeans(n_clusters=best_k, n_init=10, random_state=RANDOM_STATE)
    kmeans.fit(X_scaled)

    # Guardar artefactos y métricas
    joblib.dump(scaler, os.path.join(DATA_DIR, "scaler.joblib"))
    joblib.dump(kmeans, os.path.join(DATA_DIR, "kmeans.joblib"))
    metrics_df["K_selected"] = (metrics_df["K"] == best_k).astype(int)
    metrics_df.to_csv(os.path.join(DATA_DIR, "metrics_k.csv"), index=False)

    # Estadísticas por clúster (para interpretar)
    # Usamos toda la muestra X_all
    labels = kmeans.predict(X_scaled)
    df = pd.DataFrame(X_all, columns=["VV_dB", "VH_dB", "PR_dB"])
    df["cluster"] = labels

    stats = df.groupby("cluster").agg(
        VV_mean=("VV_dB", "mean"), VV_std=("VV_dB", "std"),
        VH_mean=("VH_dB", "mean"), VH_std=("VH_dB", "std"),
        PR_mean=("PR_dB", "mean"), PR_std=("PR_dB", "std"),
        count=("cluster", "count")
    ).reset_index()

    # clusters 0..K-1 -> para ArcGIS preferimos 1..K en la interpretación
    stats["class_label"] = stats["cluster"] + 1
    stats = stats[["class_label", "count", "VV_mean", "VV_std", "VH_mean", "VH_std", "PR_mean", "PR_std"]]
    stats.to_csv(os.path.join(DATA_DIR, "cluster_stats.csv"), index=False)

    print("\n== Artefactos guardados ==")
    print(os.path.join(DATA_DIR, "scaler.joblib"))
    print(os.path.join(DATA_DIR, "kmeans.joblib"))
    print(os.path.join(DATA_DIR, "metrics_k.csv"))
    print(os.path.join(DATA_DIR, "cluster_stats.csv"))

if __name__ == "__main__":
    main()

In [ ]:
import joblib
kmeans = joblib.load("C:/SPA_SEI/PROYECTOS/IIAP/Sedimentos/sentinel_1/data_cut_dB/kmeans.joblib")

In [ ]:
kmeans

In [ ]:
print("Número de clusters:", kmeans.n_clusters)
print("Centroides (en espacio normalizado):")
print(kmeans.cluster_centers_)